In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from sklearn.mixture import GaussianMixture
import numpy as np
import pandas as pd


def bin_by_relevance_with_stats(
    df,
    score_col,
    high_q=0.67,
    low_q=0.33,
    n_components=3,
    random_state=42,
    covariance_type="full",
):
    """
    Split rows into 3 learned score-based clusters (high / mid / low) using
    a Gaussian Mixture Model on the score column only.

    This is a drop-in replacement for the quantile-based version and returns:

        bins: dict with keys {"high", "mid", "low"} -> DataFrames
        stats: dict with per-bin statistics
        thresholds: dict with threshold-like metadata

    Notes:
    - high_q and low_q are kept in the signature for compatibility, but are not
      used for clustering here.
    - Clusters are ordered by their mean score:
          highest mean -> "high"
          middle mean  -> "mid"
          lowest mean  -> "low"
    """
    if df is None or len(df) == 0:
        empty = df.copy()
        bins = {"high": empty, "mid": empty, "low": empty}
        stats = {
            "high": {"count": 0, "pct": 0.0, "mean_score": 0.0, "min_score": 0.0, "max_score": 0.0},
            "mid":  {"count": 0, "pct": 0.0, "mean_score": 0.0, "min_score": 0.0, "max_score": 0.0},
            "low":  {"count": 0, "pct": 0.0, "mean_score": 0.0, "min_score": 0.0, "max_score": 0.0},
        }
        thresholds = {
            "high_threshold": None,
            "low_threshold": None,
            "high_q": high_q,
            "low_q": low_q,
            "method": "gmm_score_only",
        }
        return bins, stats, thresholds

    df2 = df.copy()

    # Prepare 1D score array
    scores = df2[score_col].to_numpy(dtype=np.float32).reshape(-1, 1)

    # If there are too few unique values, GMM can become unstable.
    # Fall back to quantile split in that case.
    # unique_scores = np.unique(scores.ravel())
    # if len(unique_scores) < min(n_components, 3):
    #     hi = float(np.quantile(scores.ravel(), high_q))
    #     lo = float(np.quantile(scores.ravel(), low_q))

    #     high_df = df2[df2[score_col] >= hi].copy()
    #     mid_df  = df2[(df2[score_col] < hi) & (df2[score_col] > lo)].copy()
    #     low_df  = df2[df2[score_col] <= lo].copy()

    #     total = len(df2)

    #     def _stats(subdf):
    #         if len(subdf) == 0:
    #             return {
    #                 "count": 0,
    #                 "pct": 0.0,
    #                 "mean_score": 0.0,
    #                 "min_score": 0.0,
    #                 "max_score": 0.0,
    #             }
    #         vals = subdf[score_col].astype(float)
    #         return {
    #             "count": int(len(subdf)),
    #             "pct": 100.0 * len(subdf) / total,
    #             "mean_score": float(vals.mean()),
    #             "min_score": float(vals.min()),
    #             "max_score": float(vals.max()),
    #         }

    #     bins = {"high": high_df, "mid": mid_df, "low": low_df}
    #     stats = {
    #         "high": _stats(high_df),
    #         "mid": _stats(mid_df),
    #         "low": _stats(low_df),
    #     }
    #     thresholds = {
    #         "high_threshold": hi,
    #         "low_threshold": lo,
    #         "high_q": high_q,
    #         "low_q": low_q,
    #         "method": "quantile_fallback_low_unique_scores",
    #     }
    #     return bins, stats, thresholds

    # Fit GMM
    gmm = GaussianMixture(
        n_components=n_components,
        random_state=random_state,
        covariance_type=covariance_type,
        n_init=10,
    )
    gmm.fit(scores)

    cluster_ids = gmm.predict(scores)
    cluster_probs = gmm.predict_proba(scores)

    df2["_cluster_id"] = cluster_ids
    df2["_cluster_confidence"] = cluster_probs.max(axis=1)

    # Order raw cluster ids by mean score descending
    cluster_mean_scores = (
        df2.groupby("_cluster_id")[score_col]
        .mean()
        .sort_values(ascending=False)
    )

    ordered_cluster_ids = list(cluster_mean_scores.index)

    # Keep same high/mid/low names
    target_names = ["high", "mid", "low"]
    cluster_name_map = {
        cid: name for cid, name in zip(ordered_cluster_ids, target_names)
    }

    df2["_cluster_name"] = df2["_cluster_id"].map(cluster_name_map)

    # In case n_components > 3, collapse extras into mid
    df2["_cluster_name"] = df2["_cluster_name"].fillna("mid")

    high_df = df2[df2["_cluster_name"] == "high"].copy()
    mid_df  = df2[df2["_cluster_name"] == "mid"].copy()
    low_df  = df2[df2["_cluster_name"] == "low"].copy()

    total = len(df2)

    def _stats(subdf):
        if len(subdf) == 0:
            return {
                "count": 0,
                "pct": 0.0,
                "mean_score": 0.0,
                "min_score": 0.0,
                "max_score": 0.0,
            }
        vals = subdf[score_col].astype(float)
        return {
            "count": int(len(subdf)),
            "pct": 100.0 * len(subdf) / total,
            "mean_score": float(vals.mean()),
            "min_score": float(vals.min()),
            "max_score": float(vals.max()),
        }

    stats = {
        "high": _stats(high_df),
        "mid": _stats(mid_df),
        "low": _stats(low_df),
    }

    # These are not true quantile thresholds anymore, but useful threshold-like
    # metadata for compatibility / reporting.
    high_threshold = stats["high"]["min_score"] if stats["high"]["count"] > 0 else None
    low_threshold = stats["low"]["max_score"] if stats["low"]["count"] > 0 else None

    thresholds = {
        "high_threshold": high_threshold,
        "low_threshold": low_threshold,
        "high_q": high_q,
        "low_q": low_q,
        "method": "gmm_score_only",
        "n_components": n_components,
        "cluster_order_by_mean": {
            "high": stats["high"]["mean_score"],
            "mid": stats["mid"]["mean_score"],
            "low": stats["low"]["mean_score"],
        },
    }

    # Drop helper columns from returned bins if you want cleaner outputs
    helper_cols = ["_cluster_id", "_cluster_confidence", "_cluster_name"]
    high_df = high_df.drop(columns=helper_cols, errors="ignore")
    mid_df  = mid_df.drop(columns=helper_cols, errors="ignore")
    low_df  = low_df.drop(columns=helper_cols, errors="ignore")

    bins = {"high": high_df, "mid": mid_df, "low": low_df}
    return bins, stats, thresholds

In [ ]:
def dedup_sentences(sentences, min_chars=10):
    # This function:
    # 1) cleans each sentence (strip spaces, collapse multiple spaces)
    # 2) removes very short sentences (length < min_chars)
    # 3) removes duplicates ignoring case
    # 4) returns the cleaned unique sentences in original order

    # explain me out and seen
    # out starts as an empty list that will hold the final cleaned sentences.
    # seen starts as an empty set that will track the lowercase versions of sentences we've already added to out, to avoid duplicates.

    out, seen = [], set()
    for s in sentences:
        s = "" if s is None else str(s).strip()
        s = " ".join(s.split())
        if len(s) < min_chars:
            continue
        key = s.lower()
        if key in seen:
            continue
        seen.add(key)
        out.append(s)
    return out

def fallback_summarizer(prompt, max_sentences=4):
    """
    Very simple summarizer that works without an LLM.

    It assumes your prompt includes bullet lines like:
      - sentence 1
      - sentence 2
      - sentence 3

    It extracts those bullet sentences, removes duplicates, and returns the
    first few as a "summary" (joined into one paragraph).
    """
    
    lines = prompt.splitlines()
    candidates = []
    for ln in lines:
        ln = ln.strip()
        if ln.startswith("- "):
            candidates.append(ln[2:].strip())
    candidates = dedup_sentences(candidates)
    return " ".join(candidates[:max_sentences])

In [ ]:
def prompt_high(sentences):
    return (
        "TASK: Write a concise summary focused on the user's PRIMARY needs.\n"
        "RULES:\n"
        "- Use 2–4 sentences.\n"
        "- Write prose (no bullets in output).\n"
        "- Prefer concrete points.\n"
        "SENTENCES:\n"
        + "\n".join([f"- {s}" for s in sentences])
    )

def prompt_mid(sentences, S_H):
    return (
        "TASK: Add SECONDARY details from the sentences below.\n"
        "CONTEXT (already covered):\n"
        f"{S_H}\n\n"
        "RULES:\n"
        "- Do NOT repeat the same points.\n"
        "- Add complementary details only.\n"
        "- Use 1–3 sentences.\n"
        "SENTENCES:\n"
        + "\n".join([f"- {s}" for s in sentences])
    )

def prompt_low(sentences):
    return (
        "TASK: Extract only caveats, edge cases, or warnings (if any).\n"
        "RULES:\n"
        "- Use at most 1–2 short sentences.\n"
        "SENTENCES:\n"
        + "\n".join([f"- {s}" for s in sentences])
    )

def prompt_stitch(S_H, S_M, S_L=""):
    return (
        "TASK: Combine the parts into ONE coherent final summary.\n"
        "RULES:\n"
        "- No repetition.\n"
        "- Keep it short.\n"
        "- Prefer a single paragraph.\n\n"
        "HIGH SUMMARY:\n"
        f"{S_H}\n\n"
        "MID SUMMARY:\n"
        f"{S_M}\n\n"
        "LOW NOTES (optional):\n"
        f"{S_L}\n"
    )

In [ ]:
def basic_prompt(sentences):
    return " ".join(sentences)
 
def contextual_rewrite_baseline(
    selected_df,
    text_col="review_text",
    score_col="utility",    # "utility" (stochastic) or "Rel" (MMR)
    high_q=0.67,
    low_q=0.33,
    max_high=8,
    max_mid=8,
    max_low=6,
    use_low=False,
    summarizer=None,
):
    if summarizer is None:
        summarizer = fallback_summarizer

    if selected_df is None or len(selected_df) == 0:
        return {
            "S_H": "",
            "S_M": "",
            "S_L": "",
            "final": "",
            "bin_stats": {},
            "thresholds": {},
            "bins": {"high": None, "mid": None, "low": None},
            "prompts": {"high": "", "mid": "", "low": "", "final": ""},
        }

    # Sort by relevance so we take the strongest sentences from each bin
    df_sorted = selected_df.sort_values(score_col, ascending=False).copy()

    # Bin into high/mid/low
    bins, stats, thresholds = bin_by_relevance_with_stats(
        df_sorted,
        score_col=score_col,
        high_q=high_q,
        low_q=low_q,
    )

    # Collect sentences (dedup + truncate)
    high_sents = dedup_sentences(bins["high"][text_col].tolist())[:max_high]
    mid_sents  = dedup_sentences(bins["mid"][text_col].tolist())[:max_mid]
    low_sents  = dedup_sentences(bins["low"][text_col].tolist())[:max_low]

    # 1) Summarize high bin
    pH = basic_prompt(high_sents)
    S_H = summarizer(pH)

    # 2) Summarize mid bin conditioned on S_H
    S_M = ""
    pM = ""
    if len(mid_sents) > 0:
        pM = basic_prompt(mid_sents)
        S_M = summarizer(pM)

    # 3) Optional low bin for caveats
    S_L = ""
    pL = ""
    if use_low and len(low_sents) > 0:
        pL = basic_prompt(low_sents)
        S_L = summarizer(pL)

    # 4) Stitch final
    pF = basic_prompt([S_H, S_M, S_L])
    final = summarizer(pF)

    return {
        "S_H": S_H,
        "S_M": S_M,
        "S_L": S_L,
        "final": final,
        "bin_stats": stats,
        "thresholds": thresholds,
        "bins": {"high": bins["high"], "mid": bins["mid"], "low": bins["low"]},
        "prompts": {"high": pH, "mid": pM, "low": pL, "final": pF},
    }

In [ ]:
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
MODEL_NAME = "facebook/bart-large-cnn"
# MODEL_NAME = "google/pegasus-xsum"

device = 0 if torch.cuda.is_available() else -1
summarizer_pipe = pipeline(
    task="summarization",
    model=MODEL_NAME,
    device=device
)

print("Using device:", "GPU" if device == 0 else "CPU")

In [ ]:
def llm_summarizer(prompt: str,
                  max_new_tokens: int = 90,
                  min_new_tokens: int = 25,
                  do_sample: bool = False):
    """
    HuggingFace summarization wrapper: prompt -> summary text

    Works well when prompt contains bullet points.
    """
    prompt = prompt.strip()

    # Transformers summarization pipeline uses max_length/min_length in tokens
    out = summarizer_pipe(
        prompt,
        max_length=max_new_tokens,
        min_length=min_new_tokens,
        do_sample=do_sample,
        truncation=True,
    )
    return out[0]["summary_text"].strip()

In [ ]:
out_gumbel = pd.read_csv("../data/extracted_sentences_gumbel_extractor.csv")
out_mmr = pd.read_csv("../data/extracted_sentences_deterministic_MMR_Style.csv")

In [ ]:
rewrite = contextual_rewrite_baseline(
    selected_df=out_gumbel,
    text_col="review_text",
    score_col="utility",
    use_low=True,
    summarizer=llm_summarizer,
)

# selected sentences in each bin as prompts for the summary rewriter:
print(rewrite["prompts"]["high"], "\n")
print(rewrite["prompts"]["mid"], "\n")
print(rewrite["prompts"]["low"], "\n")

# final summaries for each bin:
print("S_H:\n", rewrite["S_H"])
print("\nS_M:\n", rewrite["S_M"])
print("\nS_L:\n", rewrite["S_L"])

# outputting statistics: (most)
# add quantile binning numbers to the summary bins. size of the cluster (no. of reviews) in each bin. 



In [ ]:
def make_bart_friendly_stitch_text(bin_payloads):
    pieces = []
    for name in ["high", "mid", "low"]:
        p = bin_payloads.get(name, {})
        s = p.get("stats", {})
        summary = str(p.get("summary", "")).strip()
        if not summary:
            continue

        if name == "high":
            prefix = "The most aligned cluster"
        elif name == "mid":
            prefix = "A moderately aligned cluster"
        else:
            prefix = "A less aligned cluster"

        pieces.append(
            f"{prefix}, representing {s['pct']:.1f}% of selected reviews "
            f"({s['count']} reviews), has an average relevance score of {s['mean_score']:.3f}. "
            f"{summary}"
        )

    pieces.append(
        "Overall, the higher-relevance cluster reflects opinions closest to the user's preferences, "
        "while lower-relevance clusters provide secondary themes and caveats."
    )
    return " ".join(pieces)

bin_payloads = {
    "high": {
        "summary": rewrite["S_H"],
        "stats": rewrite["bin_stats"]["high"],
    },
    "mid": {
        "summary": rewrite["S_M"],
        "stats": rewrite["bin_stats"]["mid"],
    },
    "low": {
        "summary": rewrite["S_L"],
        "stats": rewrite["bin_stats"]["low"],
    },
}

final_input_text = make_bart_friendly_stitch_text(bin_payloads)

print(final_input_text)

The most aligned cluster, representing 30.0% of selected reviews (3 reviews), has an average relevance score of 0.086. These are great for exfoliating at home! Nice enough to use on your face too. Helps with my keratosis pillaris. Easy to wash. A moderately aligned cluster, representing 60.0% of selected reviews (6 reviews), has an average relevance score of -0.005. Best scrubbing you'll ever get. Works better than any bathing device I have ever used before. Leaves me feeling exfoliated and clean. A less aligned cluster, representing 10.0% of selected reviews (1 reviews), has an average relevance score of -0.205. These don't tear & cant have a good shower without them. They do age cause nothing lasts forever. Was so good to wash with a brand new one. Overall, the higher-relevance cluster reflects opinions closest to the user's preferences, while lower-relevance clusters provide secondary themes and caveats.


In [ ]:
final_summary = llm_summarizer(
    final_input_text,
    max_new_tokens=120,
    min_new_tokens=40,
    do_sample=False
)

print("\nFINAL STITCHED SUMMARY:\n")
print(final_summary)

These are great for exfoliating at home! Nice enough to use on your face too. Helps with my keratosis pillaris. Easy to wash. Best scrubbing you'll ever get. Works better than any bathing device I have ever used.


### As can be seen here, BART by itself doesnot look good on the stitched summary, it excludes the sentences altogether. So , we can use a 2 stage model: stage 1: only BART for each bin compression, and then a instruction fine-tuned model for stitiching as a stage 2..

In [ ]:
INSTRUCT_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(INSTRUCT_MODEL_NAME)

instruct_pipe = pipeline(
    "text-generation",
    model=INSTRUCT_MODEL_NAME,
    tokenizer=tokenizer,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

def instruct_rewriter(prompt: str,
                      max_new_tokens: int = 1024,
                      min_new_tokens: int = 60):
    messages = [
        {"role": "system", "content": "You rewrite text clearly, faithfully, and concisely."},
        {"role": "user", "content": prompt.strip()},
    ]
    model_input = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    out = instruct_pipe(
        model_input,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        truncation=True,
        return_full_text=False,
    )
    return out[0]["generated_text"].strip()

In [ ]:
def make_instruct_stitch_prompt(bin_payloads, score_col="utility"):
    high = bin_payloads["high"]
    mid  = bin_payloads["mid"]
    low  = bin_payloads["low"]

    low_summary = low["summary"].strip() if low["summary"] else "No important low-relevance caveats."

    return f"""
            Task: Write a final personalized review summary using the cluster data below.

            Rules:
            1. Write exactly 4 sentences.
            2. Sentence 1 must describe the HIGH cluster.
            3. Sentence 2 must describe the MID cluster.
            4. Sentence 3 must describe the LOW cluster.
            5. Sentence 4 must give an overall recommendation.
            6. Mention the percentage in sentences 1, 2, and 3.
            7. HIGH means closest to the user's preferences.
            8. Do not use bullet points.
            9. Do not copy the input text verbatim.

            Input:
            HIGH | pct={high['stats']['pct']:.1f}% | count={high['stats']['count']} | mean_{score_col}={high['stats']['mean_score']:.3f} | summary={high['summary']}
            MID | pct={mid['stats']['pct']:.1f}% | count={mid['stats']['count']} | mean_{score_col}={mid['stats']['mean_score']:.3f} | summary={mid['summary']}
            LOW | pct={low['stats']['pct']:.1f}% | count={low['stats']['count']} | mean_{score_col}={low['stats']['mean_score']:.3f} | summary={low_summary}

            Output:
            """.strip()

# def make_instruct_stitch_prompt(bin_payloads, score_col="utility"):
#     high = bin_payloads["high"]
#     mid  = bin_payloads["mid"]
#     low  = bin_payloads["low"]

#     return f"""
#             You are writing a personalized review summary from relevance-based clusters.

#             Interpretation:
#             - HIGH cluster = closest to the user's preferences
#             - MID cluster = moderately aligned opinions
#             - LOW cluster = less aligned opinions or caveats

#             Use the following cluster information to write one compact final summary.
#             Requirements:
#             - Start with HIGH
#             - Then mention MID
#             - Then mention LOW if it is non-empty
#             - Explicitly mention percentages
#             - Make it sound like a recommendation-oriented personalized summary
#             - Keep it to 4-6 sentences
#             - Do not use bullet points

#             HIGH CLUSTER
#             - count: {high['stats']['count']}
#             - percentage: {high['stats']['pct']:.1f}%
#             - mean {score_col}: {high['stats']['mean_score']:.3f}
#             - summary: {high['summary']}

#             MID CLUSTER
#             - count: {mid['stats']['count']}
#             - percentage: {mid['stats']['pct']:.1f}%
#             - mean {score_col}: {mid['stats']['mean_score']:.3f}
#             - summary: {mid['summary']}

#             LOW CLUSTER
#             - count: {low['stats']['count']}
#             - percentage: {low['stats']['pct']:.1f}%
#             - mean {score_col}: {low['stats']['mean_score']:.3f}
#             - summary: {low['summary']}

#             Now write the final stitched summary.
#             """.strip()

In [ ]:
final_prompt = make_instruct_stitch_prompt(bin_payloads, score_col="utility")
final_summary = instruct_rewriter(final_prompt)

print("\nFINAL INSTRUCT-BASED STITCHED SUMMARY:\n")
print(final_summary)

These are great for exfoliating at home! Nice enough to use on your face too. Helps with my keratosis pillaris. Easy to wash. These don't tear & cant have a good shower without them. They do age cause nothing lasts forever. Was so good to wash with a brand new one.

In [ ]:
def make_final_third_person_prompt(final_summary_text):
    return f"""
        Rewrite the following summary so it is suitable to display to users.

        Rules:
        - Use third-person generalization.
        - Remove first-person phrases like "I", "me", "my", "we".
        - Do not sound like direct review quotes.
        - Keep all statistics and percentages unchanged.
        - Preserve the overall meaning.
        - Return one polished paragraph.

        Summary:
        {final_summary_text}

        Rewritten summary:
        """.strip()

third_person_prompt = make_final_third_person_prompt(final_input_text.strip())
final_polished_summary = instruct_rewriter(third_person_prompt,max_new_tokens=180)

print(final_polished_summary)

Overall, the higher-relevance cluster reflects opinions closest to the user's preferences, while the lower-relevance clusters provide secondary themes and caveats. Overall, the higher-relevance cluster reflects opinions closest to the user's preferences, while the lower-relevance clusters provide secondary themes and caveats.

In [ ]:
final_polished_summary

"The study analyzed three clusters of customer feedback regarding a product:\n\n- **Cluster 1:** Representing 30.0% of the reviews (3 out of 10), this group has an average relevance score of 0.086. Users appreciate its effectiveness for home exfoliation and its suitability for facial use. It also aids in treating keratosis pilaris. The product is easy to clean. This cluster aligns closely with user preferences.\n\n- **Cluster 2:** Comprising 60.0% of the reviews (6 out of 10), this group has an average relevance score of -0.005. Customers find it highly effective for intense exfoliation and superior to other devices they've tried. It leaves them feeling thoroughly exfoliated and clean after use. However, some users note that it may leave skin looking slightly"